# 01 — Data Preparation and Data Quality

This notebook loads the official NGCP **Luzon Actual Load** worksheet, converts the 24 source-hour columns into a long hourly time series, assigns the three study periods, and checks whether the data are suitable for modeling.

### What to look for
- Every study date should contain exactly **24 hourly observations**.
- Missing, duplicate, or negative demand values should be zero (or explicitly investigated).
- Period coverage should match **2017–2019**, **2020–2021**, and **2022–2024**.
- The source workbook labels columns as **Hour No. 1–24**. Because the workbook itself does not define clock-time labels, this project reports **NGCP Hour No.** in findings. The internal `hour` field is only a zero-based analysis index (`Hour No. - 1`).

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src" / "data_preparation.py").is_file():
    project_root = project_root.parent
if not (project_root / "src" / "data_preparation.py").is_file():
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import PERIOD_ORDER
figures_dir = project_root / "figures"
results_dir = project_root / "results"
figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

from src.data_preparation import prepare_dataset, data_quality_report
from src.config import NGCP_DATE_COLUMN, NGCP_HEADER_ROW, NGCP_LUZON_SHEET

input_path = project_root / "data/raw/Hourly Demand per Grid.xlsx"
df = prepare_dataset(
    input_path,
    sheet_name=NGCP_LUZON_SHEET,
    header=NGCP_HEADER_ROW,
    date_col=NGCP_DATE_COLUMN,
    study_only=True,
)
quality = data_quality_report(df)
quality

In [ ]:
coverage = (
    df.groupby("period", observed=True)
      .agg(start_date=("date", "min"), end_date=("date", "max"),
           days=("date", "nunique"), observations=("demand_mw", "size"))
      .reindex(PERIOD_ORDER)
      .reset_index()
)
coverage

In [ ]:
hour_mapping = pd.DataFrame({
    "NGCP Hour No.": range(1, 25),
    "internal hour index": range(24),
})
hour_mapping

**Hour-label convention used in this study:** the workbook says only **“Hour No.”** and labels the 24 demand columns from 1 to 24. We therefore avoid unsupported statements such as “Hour 14 = exactly 1:00 PM.” The model uses a zero-based index internally, but tables and plots intended for interpretation use the original **NGCP Hour No. 1–24**.

In [ ]:
df.head(30)

In [ ]:
out = project_root / "data/processed/luzon_hourly_clean.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
quality.to_csv(results_dir / "data_quality_report.csv", index=False)
coverage.to_csv(results_dir / "period_coverage.csv", index=False)
print(f"Saved {len(df):,} hourly observations to {out}")
print(f"Complete days: {df['date'].nunique():,}; expected hourly rows: {df['date'].nunique()*24:,}")

In [ ]:
q = dict(zip(quality["metric"], quality["value"]))
checks_ok = (
    q.get("missing_demand_values", 1) == 0
    and q.get("duplicate_datetimes", 1) == 0
    and q.get("negative_demand_values", 1) == 0
    and q.get("complete_24_hour_days", -1) == q.get("unique_dates", -2)
)
print("DATA QUALITY RESULT:", "PASS" if checks_ok else "REVIEW REQUIRED")
if checks_ok:
    print("All study dates have 24 hourly slots, with no missing, duplicate, or negative demand observations.")